# Step 4: XGBoost Dual-Objective Matcher
Trains two XGBoost models:
- **Regressor**: predicts detour distance (km)
- **Classifier**: predicts acceptance probability

Includes grid search hyperparameter tuning, 5-fold cross-validation, and saves feature importance + ROC curve plots.

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [1]:
# Run from project root
import os
if os.path.basename(os.getcwd()) in ['models', 'simulation', 'data']:
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
print('Working directory:', os.getcwd())

# Enable inline plots for notebook display
%matplotlib inline

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [2]:
import pandas as pd
import numpy as np

from src.models.xgb_matching import MultiObjectiveMatcher
print("Imported MultiObjectiveMatcher from src/models/xgb_matching.py")

Imported MultiObjectiveMatcher from src/models/xgb_matching.py


In [3]:
# Run the XGBoost dual-objective matcher
print("Loading simulated candidate pairs...")
df = pd.read_parquet("data/simulated/candidate_pairs.parquet")
matcher = MultiObjectiveMatcher()
X_test, yr_test, yc_test, pred_detour, pred_match = matcher.train(df)

Loading simulated candidate pairs...
Training on 3512 samples with 9 features.
Driver-level split: 2828 train pairs, 684 test pairs.
No driver_id overlap between train and test: True

Running grid search for classifier hyperparameters...


Best classifier params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50}
Class balance — pos: 1255, neg: 1573, scale_pos_weight: 1.253

Running 5-fold cross-validation...


Classifier CV Accuracy: 0.9353 (+/- 0.1137)
  Fold scores: [0.7085 0.9894 1.     1.     0.9788]
Regressor CV MAE: 2.8024 (+/- 1.6356)
  Fold scores: [0.9393 1.0229 2.7929 4.4428 4.8143]

Training final models with best hyperparameters...
Regression MAE (Detour km): 3.32
Classification Accuracy: 0.95
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.91      0.95       382
           1       0.90      1.00      0.95       302

    accuracy                           0.95       684
   macro avg       0.95      0.96      0.95       684
weighted avg       0.96      0.95      0.95       684


Regression RMSE (Detour km): 7.7142
Classification F1: 0.9467
Classification ROC-AUC: 0.9774
Saved figures/feature_importance.png
Saved figures/roc_curve.png

Running feature ablation study (ROC-AUC drop)...


ROC-AUC drop when feature removed (higher = more important):
  overlap_ratio: +0.0127
  shareability_count: +0.0005
  driver_approx_fare: +0.0003
  fare_ratio: +0.0003
  driver_orig_dist: +0.0000
  pickup_hour: +0.0000
  pickup_dow: +0.0000
  rider_approx_fare: -0.0001
  time_diff_mins: -0.0016
Saved figures/ablation_study.png

Generating learning curves...
Saved figures/learning_curve.png

Running system-level bipartite matching evaluation...


/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
2 fits failed out of a total of 25.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/xgboost/sklearn.py", line 1761, in fit
    raise Valu


System-Level Results (Bipartite Matching on Test Set):
  Unique test drivers  : 426
  Optimal matches      : 423
  Match rate           : 99.3%
  Avg driver detour    : 14.51 km
  Avg driver earnings  : +$5.70 per ride
  Avg rider savings    : -$5.70 per ride
  Avg CO2 saved        : 3072 g per match


Saved figures/system_tradeoff_curve.png
